In [ ]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 8.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re
from uuid import uuid4
from unidecode import unidecode

# =========================
# 1. Đường dẫn file
# =========================
fake_path = "fake_news_publish_date_completed_best_effort.xlsx"
real_path = "only_real_fixed_publish_date_all_completed.xlsx"

fake_df = pd.read_excel(fake_path)
real_df = pd.read_excel(real_path)

# =========================
# 2. Chuẩn hoá tên cột
# =========================
fake_df.columns = [c.strip().lower() for c in fake_df.columns]
real_df.columns = [c.strip().lower() for c in real_df.columns]

fake_df = fake_df.loc[:, ~fake_df.columns.str.contains("^unnamed")]
real_df = real_df.loc[:, ~real_df.columns.str.contains("^unnamed")]

# =========================
# 3. Đảm bảo đủ cột cần thiết
# =========================
required_cols = [
    "id", "title", "content", "source", "url",
    "publish_date", "topic", "label", "explanation"
]

for col in required_cols:
    if col not in fake_df.columns:
        fake_df[col] = np.nan
    if col not in real_df.columns:
        real_df[col] = np.nan

# =========================
# 4. Tạo id nếu thiếu
# =========================
def ensure_id(df):
    df["id"] = df["id"].fillna("").astype(str)
    mask = df["id"].str.strip() == ""
    df.loc[mask, "id"] = [str(uuid4()) for _ in range(mask.sum())]
    return df

fake_df = ensure_id(fake_df)
real_df = ensure_id(real_df)

# =========================
# 5. Chuẩn hoá label
# =========================
def normalize_label(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().upper()

    if "FAKE" in x:
        return "FAKE"
    if "REAL" in x:
        return "REAL"

    return x

fake_df["label"] = fake_df["label"].apply(normalize_label)
real_df["label"] = real_df["label"].apply(normalize_label)

# =========================
# 6. Chuẩn hoá source
# =========================
source_map = {
    "vnexpress": "vnexpress",
    "vn express": "vnexpress",

    "tuoitre": "tuoitre",
    "tuoi tre": "tuoitre",
    "tuổi trẻ": "tuoitre",

    "thanhnien": "thanhnien",
    "thanh nien": "thanhnien",
    "thanh niên": "thanhnien",

    "vietnamnet": "vietnamnet",

    "dantri": "dantri",
    "dan tri": "dantri",
    "dân trí": "dantri",

    "laodong": "laodong",
    "lao dong": "laodong",
    "lao động": "laodong",

    "nld": "nld",
    "nguoi lao dong": "nld",
    "người lao động": "nld",

    "zing": "zingnews",
    "zingnews": "zingnews",

    "tienphong": "tienphong",
    "tien phong": "tienphong",
    "tiền phong": "tienphong",

    "plo": "phapluat",
    "phap luat": "phapluat",
    "pháp luật": "phapluat",

    "vov": "vov",
    "vtv": "vtv",

    "chinhphu": "chinhphu",
    "chính phủ": "chinhphu",

    "moh": "moh",
    "bo y te": "moh",
    "bộ y tế": "moh",

    "who": "who",

    "tingia": "tingia",
    "vnfactcheck": "vnfactcheck",

    "facebook": "facebook",
    "fb": "facebook",
    "tiktok": "tiktok",
    "youtube": "youtube",

    "daikynguyen": "daikynguyen",
    "dai ky nguyen": "daikynguyen",
    "đại kỷ nguyên": "daikynguyen",
}

def normalize_source(x):
    if pd.isna(x):
        return "unknown"

    x = str(x).strip().lower()

    if x in ["", "nan", "none", "unknown"]:
        return "unknown"

    x = re.sub(r"https?://", "", x)
    x = re.sub(r"www\.", "", x)
    x = x.strip("/")
    x = x.split("/")[0]

    x = x.replace(".com.vn", "")
    x = x.replace(".com", "")
    x = x.replace(".vn", "")
    x = x.replace(".net", "")
    x = x.replace(".org", "")
    x = x.replace(".gov", "")

    x_ascii = unidecode(x)

    for key, value in source_map.items():
        if key in x or unidecode(key) in x_ascii:
            return value

    return x_ascii.replace(" ", "_").strip()

fake_df["source"] = fake_df["source"].apply(normalize_source)
real_df["source"] = real_df["source"].apply(normalize_source)

# =========================
# 7. Chuẩn hoá topic
# =========================
topic_map = {
    "sức khỏe": "health",
    "suc khoe": "health",
    "y tế": "health",
    "y te": "health",
    "health": "health",
    "covid": "health",
    "covid-19": "health",
    "infectious_disease": "health",
    "oncology": "health",
    "pediatrics": "health",

    "giáo dục": "education",
    "giao duc": "education",
    "education": "education",

    "kinh doanh": "economy",
    "business": "economy",
    "economy": "economy",
    "finance": "economy",
    "banking": "economy",
    "tài chính": "economy",
    "tai chinh": "economy",

    "xã hội": "society",
    "xa hoi": "society",
    "society": "society",

    "chính trị": "politics",
    "chinh tri": "politics",
    "politics": "politics",
    "government": "politics",

    "công nghệ": "technology",
    "cong nghe": "technology",
    "technology": "technology",
    "cybersecurity": "technology",

    "giải trí": "entertainment",
    "giai tri": "entertainment",
    "entertainment": "entertainment",

    "lừa đảo": "scam",
    "lua dao": "scam",
    "scam": "scam",

    "thể thao": "sports",
    "the thao": "sports",
    "sports": "sports",

    "môi trường": "environment",
    "moi truong": "environment",
    "environment": "environment",

    "nông nghiệp": "agriculture",
    "nong nghiep": "agriculture",
    "agriculture": "agriculture",

    "pháp luật": "law",
    "phap luat": "law",
    "law": "law"
}

def normalize_topic(x):
    if pd.isna(x):
        return "unknown"

    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    x_ascii = unidecode(x)

    for key, value in topic_map.items():
        if key in x or unidecode(key) in x_ascii:
            return value

    return x_ascii.replace(" ", "_")

fake_df["topic"] = fake_df["topic"].apply(normalize_topic)
real_df["topic"] = real_df["topic"].apply(normalize_topic)

# =========================
# 8. Chuẩn hoá publish_date
# =========================
def normalize_date(x):
    if pd.isna(x):
        return "UNKNOWN"

    x = str(x).strip()

    if x.upper() in ["UNKNOWN", "NAN", "NONE", ""]:
        return "UNKNOWN"

    dt = pd.to_datetime(x, dayfirst=True, errors="coerce")

    if pd.isna(dt):
        return "UNKNOWN"

    return dt.strftime("%d/%m/%Y")

fake_df["publish_date"] = fake_df["publish_date"].apply(normalize_date)
real_df["publish_date"] = real_df["publish_date"].apply(normalize_date)

# =========================
# 9. Làm sạch text
# =========================
def clean_text(x):
    if pd.isna(x):
        return ""

    x = str(x)
    x = re.sub(r"<.*?>", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

for col in ["title", "content", "source", "url", "explanation"]:
    fake_df[col] = fake_df[col].apply(clean_text)
    real_df[col] = real_df[col].apply(clean_text)

# =========================
# 10. Merge REAL + FAKE
# =========================
merged_df = pd.concat([fake_df, real_df], ignore_index=True)

# =========================
# 11. Xoá duplicate
# =========================
before = len(merged_df)

merged_df = merged_df.drop_duplicates(
    subset=["title", "content"],
    keep="first"
)

after = len(merged_df)

print("Số dòng trước khi xoá duplicate:", before)
print("Số dòng sau khi xoá duplicate:", after)
print("Số duplicate đã xoá:", before - after)

# =========================
# 12. Tạo cột text chính
# =========================
merged_df["text"] = (
    merged_df["title"].fillna("") + " " +
    merged_df["content"].fillna("")
).str.strip()

# =========================
# 13. Sắp xếp lại cột
# =========================
final_cols = [
    "id",
    "title",
    "content",
    "text",
    "source",
    "url",
    "publish_date",
    "topic",
    "label",
    "explanation"
]

for col in final_cols:
    if col not in merged_df.columns:
        merged_df[col] = ""

merged_df = merged_df[final_cols]

# =========================
# 14. Kiểm tra nhanh
# =========================
print("\nLabel distribution:")
print(merged_df["label"].value_counts())

print("\nTopic distribution:")
print(merged_df["topic"].value_counts())

print("\nSource distribution:")
print(merged_df["source"].value_counts())

print("\nMissing publish_date:")
print((merged_df["publish_date"] == "UNKNOWN").sum())

# =========================
# 15. Xuất file
# =========================
output_path = "final_merged_news_dataset_cleaned.xlsx"
merged_df.to_excel(output_path, index=False)

print("\nDONE")
print("Saved:", output_path)

Số dòng trước khi xoá duplicate: 15697
Số dòng sau khi xoá duplicate: 14909
Số duplicate đã xoá: 788

Label distribution:
label
REAL    9639
FAKE    5270
Name: count, dtype: int64

Topic distribution:
topic
health           10903
education         2181
politics           583
society            506
economy            237
sports             146
entertainment      129
law                 87
technology          87
tin_tuc             26
scam                 9
travel               7
wellness             4
agriculture          2
environment          1
unknown              1
Name: count, dtype: int64

Source distribution:
source
vnexpress      5870
daikynguyen    4563
webmd           605
tuoitre         536
vinmec          485
               ... 
emergency         1
jsonline          1
truththeory       1
buffalonews       1
medium            1
Name: count, Length: 81, dtype: int64

Missing publish_date:
0

DONE
Saved: final_merged_news_dataset_cleaned.xlsx


In [ ]:
import pandas as pd
import re

# =========================
# 1. Đọc dataset
# =========================
file_path = "final_merged_news_dataset_cleaned.xlsx"
df = pd.read_excel(file_path)

# =========================
# 2. Đảm bảo các cột text không bị NaN
# =========================
df["title"] = df["title"].fillna("").astype(str)
df["content"] = df["content"].fillna("").astype(str)

# Nếu đã có cột text thì chuẩn hóa lại
df["text"] = df["text"].fillna("").astype(str)

# Nếu text bị rỗng thì tạo lại từ title + content
df.loc[df["text"].str.strip() == "", "text"] = (
    df["title"] + " " + df["content"]
).str.strip()

# =========================
# 3. Hàm đếm từ
# =========================
def count_words(text):
    words = re.findall(r"\w+", str(text), flags=re.UNICODE)
    return len(words)

# =========================
# 4. text_length và title_length
# =========================
df["text_length"] = df["text"].apply(count_words)
df["title_length"] = df["title"].apply(count_words)

# =========================
# 5. uppercase_ratio
# =========================
def uppercase_ratio(text):
    words = re.findall(r"\b\w+\b", str(text), flags=re.UNICODE)

    if len(words) == 0:
        return 0

    upper_words = [
        w for w in words
        if w.isupper() and len(w) > 1
    ]

    return round(len(upper_words) / len(words), 4)

df["uppercase_ratio"] = df["title"].apply(uppercase_ratio)

# =========================
# 6. exclamation_count và question_count
# =========================
df["exclamation_count"] = df["text"].apply(lambda x: str(x).count("!"))
df["question_count"] = df["text"].apply(lambda x: str(x).count("?"))

# =========================
# 7. clickbait_score
# =========================
CLICKBAIT_WORDS = [
    "sốc", "khẩn", "nóng", "cảnh báo", "bí mật", "sự thật",
    "gây sốc", "kinh hoàng", "chấn động", "rúng động",
    "bất ngờ", "giật mình", "không thể tin", "điều chưa biết",
    "vạch trần", "hé lộ", "cẩn thận", "nguy hiểm",
    "hoang mang", "xôn xao", "đáng sợ", "sửng sốt"
]

def calculate_clickbait_score(title):
    title_raw = str(title)
    title_lower = title_raw.lower()

    score = 0

    keyword_count = sum(
        1 for word in CLICKBAIT_WORDS
        if word in title_lower
    )

    score += keyword_count * 1.0
    score += title_raw.count("!") * 0.5
    score += title_raw.count("?") * 0.3
    score += uppercase_ratio(title_raw) * 3

    title_len = count_words(title_raw)

    if title_len <= 8 and keyword_count > 0:
        score += 0.5

    score = min(score / 5, 1)

    return round(score, 3)

df["clickbait_score"] = df["title"].apply(calculate_clickbait_score)

# =========================
# 8. sentiment_score
# Heuristic đơn giản, dễ giải thích
# =========================
NEGATIVE_WORDS = [
    "lừa đảo", "giả mạo", "tin giả", "sai sự thật", "xuyên tạc",
    "cảnh báo", "nguy hiểm", "thiệt hại", "mất tiền", "chiếm đoạt",
    "hoang mang", "lo lắng", "tử vong", "bệnh", "dịch",
    "khủng hoảng", "vi phạm", "xử phạt", "bắt giữ",
    "đe dọa", "rủi ro", "cấm", "bức xúc", "nghiêm trọng",
    "tai nạn", "đau khổ", "sợ hãi"
]

POSITIVE_WORDS = [
    "tích cực", "hiệu quả", "an toàn", "thành công", "hỗ trợ",
    "phục hồi", "tăng trưởng", "cải thiện", "đạt được",
    "khen thưởng", "vinh danh", "hợp tác", "phát triển",
    "ổn định", "khuyến khích", "vui vẻ", "hạnh phúc"
]

def calculate_sentiment_score(text):
    text = str(text).lower()

    neg_count = sum(text.count(word) for word in NEGATIVE_WORDS)
    pos_count = sum(text.count(word) for word in POSITIVE_WORDS)

    total = neg_count + pos_count

    if total == 0:
        return 0

    score = (pos_count - neg_count) / total

    return round(score, 3)

df["sentiment_score"] = df["text"].apply(calculate_sentiment_score)

# =========================
# 9. Kiểm tra nhanh
# =========================
new_cols = [
    "text_length",
    "title_length",
    "uppercase_ratio",
    "exclamation_count",
    "question_count",
    "clickbait_score",
    "sentiment_score"
]

print("Shape sau khi thêm feature:", df.shape)
print(df[new_cols].describe())

# =========================
# 10. Lưu file mới
# =========================
output_path = "final_dataset_with_features.xlsx"
df.to_excel(output_path, index=False)

print("Đã lưu:", output_path)

Shape sau khi thêm feature: (14882, 17)
        text_length  title_length  uppercase_ratio  exclamation_count  \
count  14882.000000  14882.000000     14882.000000       14882.000000   
mean     696.057721     13.174842         0.017844           0.061887   
std      493.190450      4.539658         0.042069           0.411187   
min       47.000000      2.000000         0.000000           0.000000   
25%      390.000000     10.000000         0.000000           0.000000   
50%      622.000000     13.000000         0.000000           0.000000   
75%      886.000000     15.000000         0.000000           0.000000   
max     7278.000000     67.000000         0.500000          17.000000   

       question_count  clickbait_score  sentiment_score  
count    14882.000000     14882.000000     14882.000000  
mean         0.608453         0.029202        -0.273505  
std          1.714081         0.058237         0.653432  
min          0.000000         0.000000        -1.000000  
25%         